Automatically go through ten pages of earnings call

https://www.fool.com/earnings-call-transcripts/

https://www.fool.com/earnings-call-transcripts/page/10/

And download the 20 earnings calls on each page.
For each earnings call:

Split the call into parts: prepared remarks, and Q&A

For each part
dentify the speakers and their text (what they say)

Store it all in two dataframes
DF1
[EarningsCall ID, CompanyName, CallDate, EarningsCallText]
Each row is an entire earnings call
DF2
[SpeakerTurnID(New round unique ID), EarningsCallID,SpeakerName, PreparedOrQA, Text]
Each row is a speaker turn in a call


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

In [ ]:
DriveCallFolder = ("/content/drive/MyDrive/Fall 2026/Text Mining/EarningsCallsFolder")

In [ ]:
# collect 200 call links
callURLs = []

for page in range(1, 11):
  if page == 1:
    url = "https://www.fool.com/earnings-call-transcripts/"
  else:
    url = ("https://www.fool.com/earnings-call-transcripts/page/" + str(page) + "/")

  response = requests.get(url)
  soup = BeautifulSoup(response.text, "html.parser")

  atags = soup.find_all("a", {"class": "text-gray-1100 group hover:no-underline!"})

  for a in atags:
    callURLs.append("https://www.fool.com" + a["href"])

  print("Page", page, ":", len(atags))

Page 1 : 20
Page 2 : 20
Page 3 : 20
Page 4 : 20
Page 5 : 20
Page 6 : 20
Page 7 : 20
Page 8 : 20
Page 9 : 20
Page 10 : 20


In [ ]:

downloadDir = Path(DriveCallFolder)

df1callRows = []
df2speakerRows = []

# earnings callIDs
for callID, url in enumerate(callURLs, start=1):
  response = requests.get(url)
  soup = BeautifulSoup(response.text, "html.parser")

  title = soup.find("h1").get_text(" ", strip=True)
  companyName = title.split(" (")[0]

  dateHeading = soup.find(id="date")
  dateText = dateHeading.find_next_sibling("p").get_text(" ", strip=True)


  transcriptHeading = soup.find(id="full-conference-call-transcript")
  paragraphs = transcriptHeading.find_next_siblings("p")

  paragraphTexts = []

  for p in paragraphs:
    paragraphTexts.append(p.get_text(" ", strip=True))

  callText = "\n\n".join(paragraphTexts)

  df1callRows.append({
      "EarningsCallID": callID,
      "CompanyName": companyName,
      "CallDate": dateText,
      "EarningsCallText": callText,
  })

  (downloadDir / f"call_{callID}.txt").write_text(callText, encoding="utf-8")

  section = "Prepared"
  currentTurn = None

  for p in paragraphs:
    text = p.get_text(" ", strip=True)
    bold = p.find("strong")

    #bold is speaker
    if bold and bold.get_text(strip=True).endswith(":"):
      label = bold.get_text(" ", strip=True)
      speakerName = label.rstrip(":").strip()
      speech = text[len(label):].strip()

      if (
        speakerName.lower() == "operator"
        and "first question" in speech.lower()
        ):
        section = "QA"

      currentTurn = {
          "SpeakerTurnID": len(df2speakerRows) + 1,
          "EarningsCallID": callID,
          "SpeakerName": speakerName,
          "PreparedOrQA": section,
          "Text": speech,
      }
      df2speakerRows.append(currentTurn)

    elif currentTurn is not None:
      # Continue the same speaker's turn across paragraphs.
      currentTurn["Text"] += "\n\n" + text

    else:
      raise ValueError(f"Missing initial speaker label: {url}")

  print(callID, companyName, dateText)


1 Replimune Oct. 28, 2026
2 Central Pacific Wednesday, Oct. 28, 2026
3 First Commonwealth Financial Wednesday, Oct. 28, 2026
4 Precision Drilling Wednesday, Oct. 28, 2026
5 Ferrovial Wed, Oct. 28, 2026
6 GSK Wednesday, Oct. 28, 2026
7 OSS Wednesday, Oct. 28, 2026
8 Amphenol Wednesday, Oct. 28, 2026
9 Jersey Mike's Wed, Oct. 28, 2026
10 Brinker Wed, Oct 28, 2026
11 Clean Harbors Wed, Oct 28, 2026
12 Hope Bancorp Wednesday, Oct. 28, 2026
13 Fortive Wednesday, Oct. 28, 2026
14 Shenandoah Wednesday, Oct. 28, 2026
15 NOV Wednesday, Oct. 28, 2026
16 Phillips 66 Wednesday, Oct. 28, 2026
17 OneSpaWorld Wednesday, Oct. 28, 2026
18 Prosperity Bancshares Wednesday, Oct. 28, 2026
19 NiSource Wednesday, Oct. 28, 2026
20 Seven Hills Realty Wednesday, Oct. 28, 2026
21 Orion Group Wednesday, Oct. 28, 2026
22 Nabors Oct. 28, 2026
23 Peapack-Gladstone Wednesday, Oct. 28, 2026
24 RenaissanceRe Wed, Oct. 28, 2026
25 Highwoods Properties Wed, Oct. 28, 2026
26 Vertiv Wednesday, Oct. 28, 2026
27 Banc of Cali

In [ ]:
# save DFs
df1 = pd.DataFrame(df1callRows)
df2 = pd.DataFrame(df2speakerRows)

df1.to_excel('/content/drive/MyDrive/Fall 2026/Text Mining/Data Frame/dataFrameCE1.xlsx', index=False)
df2.to_excel('/content/drive/MyDrive/Fall 2026/Text Mining/Data Frame/dataFrameCE2.xlsx', index=False)


In [ ]:
df1.head()

,EarningsCallID,CompanyName,CallDate,EarningsCallText
0,1,Replimune,"Oct. 28, 2026","Operator: Good day, and thank you for standing..."
1,2,Central Pacific,"Wednesday, Oct. 28, 2026","Operator: Good morning, and good afternoon lad..."
2,3,First Commonwealth Financial,"Wednesday, Oct. 28, 2026","Operator: Ladies and gentlemen, thank you for ..."
3,4,Precision Drilling,"Wednesday, Oct. 28, 2026","Operator: Good day, and thank you for standing..."
4,5,Ferrovial,"Wed, Oct. 28, 2026","Silvia Ruiz: Good afternoon, everybody. This i..."


In [ ]:
df2.head()

,SpeakerTurnID,EarningsCallID,SpeakerName,PreparedOrQA,Text
0,1,1,Operator,Prepared,"Good day, and thank you for standing by. Welco..."
1,2,1,Arleen Goldenberg,Prepared,"Thank you, operator, and good morning, everyon..."
2,3,1,Sushil Patel,Prepared,"Thank you, Arleen, and thank you for joining u..."
3,4,1,Michelle DiNapoli,Prepared,"Thank you, and good morning, everyone. I am th..."
4,5,1,Emily Hill,Prepared,"Thank you, Michelle. As a reminder, Replimune ..."
